# FinCrime Control Tower

**Team MillTech** · LangChain FSI Hackathon, London, 25 November 2026
Lukman Shoge (supervisor) · Yun Nam Chan (KYC) · Marcus Georgiou (transaction monitoring) · Alejandro Bondjale (regulatory change)

Compliance teams run KYC, transaction monitoring and regulatory change in silos, so a new rule takes weeks to reach the
controls. The Control Tower joins the three into one system: **a new regulation lands, the system turns it into policy,
KYC and monitoring behaviour changes, evals prove it, and a human signs off where it matters.**

| Agent | Built with | Owner |
|---|---|---|
| Supervisor | Deep Agents, with the regulatory agent as a sub-agent | Lukman |
| KYC onboarding (01) | LangGraph graph with human review | Nam |
| Transaction monitoring (02) | LangChain agent with short- and long-term memory and human review | Marcus |
| Regulatory change (03) | Deep agent with parallel section sub-agents | Alejandro |

The agents never call each other: they meet in one shared LangGraph Store (policy, customer risk, alerts, KYC refresh
flags, review cases). All data is synthetic, and outputs are drafts for a reviewer, not compliance advice.

## Setup

Fill in `.env` in the repo root from `.env.example`. Set `CONTROL_TOWER_OFFLINE=1` to rehearse with a scripted model
and no API key: it replays the repo's analyst labels, so it shows the plumbing, not model quality.

In [ ]:
import json, os, sys
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "control_tower").exists() else Path.cwd() / "00_control_tower"
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

OFFLINE = os.environ.get("CONTROL_TOWER_OFFLINE") == "1"
from control_tower import memory
from control_tower.pillars import kyc, reg, tm
from control_tower.policy import diff, BASELINE
from control_tower.reviews import answer_review
if OFFLINE:
    from control_tower import offline
    offline.enable()

from IPython.display import Markdown, display

def table(rows, cols):
    head = "| " + " | ".join(cols) + " |\n|" + "---|" * len(cols)
    body = "\n".join("| " + " | ".join(str(r.get(c, "")) for c in cols) + " |" for r in rows)
    display(Markdown(head + "\n" + body))

store = memory.reset_store()
CUSTOMER, USER = "CUST-026", "USR-002"
print("Offline rehearsal" if OFFLINE else "Live models, traced to LangSmith project " + os.environ.get("LANGSMITH_PROJECT", "?"))

## 1. Before PS26-7

Tomás Ferreira (CUST-026, card USR-002) says in his application that his business partner is **Viktor Harlan**, a
PEP. Name screening only checks *his* name, so today's controls rate him low and approve him.

In [ ]:
before = kyc.run_kyc(CUSTOMER, store, reg.active_policy(store))
table(before["reasons"], ["level", "text", "paragraph"])
print("Rating:", before["rating"].upper(), "| status:", before["status"])

In [ ]:
ANSWERS = {"TX-2005": {"decision": "approve", "note": "Booked a flight to a conference in Berlin."}}
tm_before = tm.run_tm(USER, store, reg.active_policy(store), reviewer_answers=ANSWERS, run_tag="before")
table(tm_before, ["transaction_id", "risk_score", "confidence", "final_decision", "decided_by"])
print("Thresholds:", tm.context_for(USER, before["rating"], reg.active_policy(store)))

## 2. PS26-7 lands

The regulatory agent (03) briefs the document section by section with parallel sub-agents. The supervisor turns the
brief into machine-readable controls, each citing its paragraph, and parks the change for a human to approve.

In [ ]:
if OFFLINE:  # the 03 deep agent needs a real model; use the document text as the brief
    brief = (ROOT.parent / "03_document_summary_agent/data/documents/PS26-7.md").read_text()
    proposal = reg.publish_policy_from_brief("PS26-7", brief, memory.DEFAULT_FIRM, store)
else:
    proposal = reg.run_reg("PS26-7", memory.DEFAULT_FIRM, store)
table(proposal["changes"], ["field", "before", "after", "paragraph"])

In [ ]:
# The compliance reviewer approves the control changes (four-eyes)
answer_review(store, proposal["case_id"], {"decision": "approve", "notes": "Checked against PS26-7."})
print("Live policy:", reg.active_policy(store).source_document)

## 3. Policy changes behaviour

Re-screening under PS26-7 picks up the stated business partner (paragraphs 4.1 and 4.3): Tomás is now high risk and
held for senior-manager sign-off (4.6). His card thresholds tighten (6.3), and two alerts in 30 days re-open his
KYC file (6.5).

In [ ]:
for item in store.search(("alerts", USER), limit=100):   # count alerts under the new policy only
    store.delete(("alerts", USER), item.key)
after = kyc.run_kyc(CUSTOMER, store, reg.active_policy(store))
table(after["reasons"], ["level", "text", "paragraph"])
print("Rating:", after["rating"].upper(), "| was", after["previous"], "| status:", after["status"])
print("Open reviews:", [r["case_id"] for r in memory.reviews(store)])

In [ ]:
tm_after = tm.run_tm(USER, store, reg.active_policy(store), reviewer_answers=ANSWERS, run_tag="after")
table(tm_after, ["transaction_id", "risk_score", "confidence", "final_decision", "decided_by"])
print("Thresholds:", tm.context_for(USER, after["rating"], reg.active_policy(store)))
print("KYC refresh flags:", memory.open_refresh_flags(store))

In [ ]:
# The senior manager signs off the EDD case
answer_review(store, f"kyc:{CUSTOMER}", {"decision": "approve", "notes": "EDD completed: source of funds verified."})
print(memory.get_customer_risk(store, CUSTOMER)["status"])

## 4. The supervisor

The same story, driven by the supervisor deep agent instead of step by step. In LangSmith the run shows the
supervisor, the regulatory deep agent and its section sub-agents, the KYC graph and the monitoring agent in one trace.
(Needs a real model; skipped in offline mode.)

In [ ]:
if not OFFLINE:
    from agents import supervisor
    result = supervisor.invoke({"messages": [{"role": "user", "content":
        "Re-screen CUST-026 under the live policy, monitor their card, then process any KYC refresh flags."}]})
    print(result["messages"][-1].text)

## 5. Evals

Code evaluators in LangSmith: policy field and citation accuracy, KYC rating correctness, reasons cited, the
refresh loop, and a safety check that no high-risk customer is approved without a human. Each pillar also keeps its own
evals (03 gold set, 02 Modules 2 and 3).

In [ ]:
from evals import control_tower_evals as ev
if OFFLINE:
    ev.run_local(["kyc-rating", "end-to-end"])
else:
    ev.run_langsmith(list(ev.SUITES), "control-tower")

## What's next

- **LangSmith Engine** on the tracing project: clusters failures from the eval runs and proposes fixes.
- **Deployment**: `langgraph.json` registers the supervisor for LangSmith Deployment as is.
- **Portal**: `uv run uvicorn portal.app:app` for the 5-minute demo screen.